In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import scipy.io as sio

data = sio.loadmat("phases/C2D.mat", squeeze_me=True)
# data = json.load(open("phases/C2D.json", "r"))


def entry(container, key):
    """container[key] of a dict (JSON) or of a MATLAB structure read by loadmat; None if absent."""
    names = container.dtype.names if isinstance(container, np.ndarray) else container
    if names is None or key not in names:
        return None
    value = container[key]
    return value.item() if isinstance(value, np.ndarray) and value.dtype == object and value.ndim == 0 else value


nx = np.asarray(data['nx'], dtype=int)
lx = np.asarray(data['lx'], dtype=float)


def concentration(name):
    """phi of a monomer type: phi_<name>, the real part of CL-FTS, or the phi structure of older files."""
    if 'phi_' + name in data.keys():
        return np.asarray(data['phi_' + name])
    if 'phi_' + name + '_real' in data.keys():
        return np.asarray(data['phi_' + name + '_real'])
    return np.asarray(entry(data['phi'], name))


phi_A = concentration('A').reshape(nx)
phi_B = concentration('B').reshape(nx)

# The angle gamma between a and b in degrees: the last of the saved angles (SCFT
# results, the checkpoint settings of fields files, or the input of older files)
input_angles = entry(data['initial_params'], 'angles') if 'initial_params' in data.keys() else None
if 'angles' in data.keys():
    gamma = float(np.atleast_1d(data['angles'])[-1])
elif 'checkpoint_settings_json' in data.keys():
    gamma = float(json.loads(str(data['checkpoint_settings_json']))['angles'][-1])
elif input_angles is not None and np.size(input_angles) > 0:
    gamma = float(np.atleast_1d(input_angles)[-1])
else:
    gamma = 90.0
    print("The file holds no lattice angles: the cell is drawn rectangular.")
a = lx[0]*np.array([1.0, 0.0])
b = lx[1]*np.array([np.cos(np.deg2rad(gamma)), np.sin(np.deg2rad(gamma))])

# Corners of the grid cells: each value fills the cell around its point, (i + 1/2)/n
# along each lattice vector as in the solvers (a along x, b in the xy plane)
fa, fb = np.meshgrid(np.arange(nx[0] + 1)/nx[0], np.arange(nx[1] + 1)/nx[1], indexing='ij')
X, Y = fa*a[0] + fb*b[0], fa*a[1] + fb*b[1]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
fig.suptitle("Concentration" , fontsize=16)
im = axes[0].pcolormesh(X, Y, phi_A, cmap=cm.jet, vmin=0.0, vmax=1.0)
im = axes[1].pcolormesh(X, Y, phi_B, cmap=cm.jet, vmin=0.0, vmax=1.0)
axes[0].set(title='phi_A', xlabel='x', ylabel='y', aspect='equal')
axes[1].set(title='phi_B', xlabel='x', ylabel='y', aspect='equal')

fig.subplots_adjust(right=1.0)
fig.colorbar(im, ax=axes.ravel().tolist())
fig.show()

# fig.savefig("2d_density.png", dpi=300, bbox_inches='tight', pad_inches=0)